# M3.5 — Chrome GUI Pilot SFT (2 Epochs + 无人值守自动导出与断开)

- **运行前只需一步**：右上角切换运行时为 **A100 GPU**，然后直接点 **代码执行程序 → 全部运行（Run all）**。
- **零误烧保护**：如果误开在 CPU 运行时，启动后第 1 秒会直接报错拦截，不会进入安装或等待。
- **全自动无人值守闭环**：校验 6 份数据包 SHA-256 与防泄漏分割 → 安装锁版训练栈 → `Qwen/Qwen3.5-2B`（2 epochs，每 50 步评估保存并写入 Drive `m35_runs/`）→ 导出 `m35-qwen35-2b-Q4_K_M.gguf`、`Q8_0` 和 `m35-mmproj-f16.gguf` 并核对 Drive 文件哈希 → 刷盘等待后**自动执行 `runtime.unassign()` 断开 A100**（无论成功或报错都会执行 `finally` 释放算力）。
- **事后单导中间权重**：若验收后想单独导出某个中间 checkpoint（如 `checkpoint-200`），可切回 **CPU** 运行时，把 `RUN_TRAINING = False`、`EXPORT_ADAPTER = "/content/drive/MyDrive/chrome-gui-pilot/m35_runs/<RUN_ID>/checkpoints/.../checkpoint-200"` 后再运行。

In [ ]:
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import random
import shutil
import stat
import subprocess
import sys
import tarfile
import time
import traceback
import zipfile

SYSTEM_PROMPT = (
    "You are a Chrome browser GUI action assistant. Given a Chrome window screenshot and a user instruction, "
    "output strictly one JSON object for the next action using [0, 1000] normalized integer coordinates. "
    "Do not output explanations or markdown."
)
BASE_MODEL = "Qwen/Qwen3.5-2B"
BASE_REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"
LLAMA_CPP_EXPORT_REVISION = "164ab018f9b7368ff8122094994c4b74e468aa0e"
POINTER_ACTIONS = {"left_click", "right_click", "double_click", "move", "scroll"}
ACTIONS = POINTER_ACTIONS | {"type", "hotkey", "wait", "done", "not_found"}

def jsonl(path):
    with path.open(encoding="utf-8-sig") as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except ValueError as error:
                    raise ValueError(f"{path}:{line_number}: invalid JSON") from error


def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def inside(root, name):
    if not isinstance(name, str) or "\\" in name or ":" in name or "\x00" in name:
        raise ValueError(f"Invalid relative path: {name!r}")
    relative = PurePosixPath(name)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe relative path: {name!r}")
    target = (root / relative).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError(f"Path escapes its bundle: {name!r}")
    return target


def extract_bundle(archive_path, destination):
    limit = min(shutil.disk_usage(destination).free - (2 << 30), 32 << 30)
    seen = set()
    total = 0

    def validate(name, size, directory):
        nonlocal total
        output = inside(destination, name)
        if output in seen or (output == destination and not directory):
            raise ValueError(f"Duplicate or invalid member: {name}")
        seen.add(output)
        total += size
        if total > limit:
            raise ValueError(f"Bundle exceeds available scratch budget: {archive_path.name}")
        return output

    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                mode = stat.S_IFMT(member.external_attr >> 16)
                if mode not in {0, stat.S_IFDIR, stat.S_IFREG}:
                    raise ValueError(f"Unsupported ZIP member: {member.filename}")
                validate(member.filename, member.file_size, member.is_dir())
            archive.extractall(destination)
    else:
        with tarfile.open(archive_path, "r:*") as archive:
            members = archive.getmembers()
            for member in members:
                if not (member.isfile() or member.isdir()):
                    raise ValueError(f"Unsupported TAR member: {member.name}")
                validate(member.name, member.size, member.isdir())
            archive.extractall(destination, members=members, filter="data")


def check_action(payload):
    if not isinstance(payload, dict) or payload.get("action") not in ACTIONS:
        raise ValueError("Unknown or missing action")
    action = payload["action"]
    allowed = {"action"}
    if action in POINTER_ACTIONS or action == "type" and "coordinate" in payload:
        allowed.add("coordinate")
        point = payload.get("coordinate")
        if not isinstance(point, list) or len(point) != 2 or any(type(value) is not int or not 0 <= value <= 1000 for value in point):
            raise ValueError("Expected a normalized integer coordinate in [0, 1000]")
    if action == "type":
        allowed.add("text")
        if not isinstance(payload.get("text"), str) or not payload["text"]:
            raise ValueError("type requires nonempty text")
    elif action == "scroll":
        allowed.update(("direction", "amount"))
        if payload.get("direction") not in {"up", "down", "left", "right"} or type(payload.get("amount")) is not int or not 1 <= payload["amount"] <= 20:
            raise ValueError("Invalid scroll direction or amount")
    elif action == "hotkey":
        allowed.add("keys")
        keys = payload.get("keys")
        if not isinstance(keys, list) or not keys or not all(isinstance(key, str) and key.strip() for key in keys):
            raise ValueError("hotkey requires a nonempty keys list")
    elif action == "wait":
        allowed.add("seconds")
        seconds = payload.get("seconds", 1)
        if type(seconds) not in {float, int} or not 0.1 <= seconds <= 10:
            raise ValueError("wait duration must be in [0.1, 10]")
    elif action == "not_found":
        allowed.add("reason")
        if "reason" in payload and not isinstance(payload["reason"], str):
            raise ValueError("not_found reason must be text")
    if payload.keys() - allowed:
        raise ValueError(f"Unexpected action fields: {sorted(payload.keys() - allowed)}")
    return action


def load_sft(root, manifest, source, image_hashes, *, training):
    path = inside(root, manifest)
    rows = []
    identifiers = set()
    for row in jsonl(path):
        sample_id = row.get("sample_id")
        try:
            if not isinstance(sample_id, str) or not sample_id or sample_id in identifiers:
                raise ValueError("Missing or repeated sample_id")
            banned = {"test", "eval_only", "holdout"} | ({"eval", "val", "dev", "validation"} if training else set())
            if str(row.get("split", "")).lower() in banned:
                raise ValueError("Held-out row cannot enter SFT")
            messages, images = row["messages"], row["images"]
            if [message["role"] for message in messages] != ["system", "user", "assistant"] or len(images) != 1:
                raise ValueError("Expected system/user/assistant and one image")
            instruction = messages[1]["content"]
            if not instruction.startswith("<image>") or instruction.count("<image>") != 1 or not instruction[7:].strip():
                raise ValueError("Expected one leading image token and a nonempty instruction")
            output = json.loads(messages[2]["content"])
            action = check_action(output)
            image = inside(root, images[0])
            if image.suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp"} or not image.is_file():
                raise ValueError(f"Missing or unsupported image: {images[0]}")
            if image not in image_hashes:
                from PIL import Image

                with Image.open(image) as decoded:
                    decoded.verify()
                image_hashes[image] = digest(image)
        except (KeyError, ValueError, TypeError, AttributeError, OSError) as error:
            raise ValueError(f"{path} ({sample_id!r}): {error}") from error
        identifiers.add(sample_id)
        rows.append((action, image_hashes[image], {
            "sample_id": f"{source}:{sample_id}",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": instruction},
                {"role": "assistant", "content": json.dumps(output, ensure_ascii=False, separators=(",", ":"))},
            ],
            "images": [str(image)],
        }))
    if not rows:
        raise ValueError(f"Empty manifest: {path}")
    return rows


def deduplicate(rows):
    unique = {}
    conflicting = set()
    for record in rows:
        _, checksum, row = record
        key = (checksum, row["messages"][1]["content"])
        previous = unique.setdefault(key, record)
        if previous[2]["messages"][2]["content"] != row["messages"][2]["content"]:
            conflicting.add(key)
    result = [record for key, record in unique.items() if key not in conflicting]
    return result, {"repeated": len(rows) - len(unique), "conflicting_pairs_removed": len(conflicting)}


def prepare(data_root, scratch, bundles, seed=42, dev_limit=None, expected_archives=None):
    scratch.mkdir(parents=True, exist_ok=False)
    training, validation = [], []
    receipt = {}
    image_hashes = {}
    train_groups = set()
    rng = random.Random(seed)
    for name, relative, train_manifest, val_manifest, train_cap in bundles:
        if not name.isascii() or not name.replace("_", "").isalnum() or name in receipt:
            raise ValueError(f"Invalid or duplicate source name: {name}")
        if any(marker in str(item).lower() for item in (relative, train_manifest, val_manifest)
               for marker in ("holdout", "screenspot", "venus")):
            raise ValueError("Independent test data is not accepted by the training preparer")
        source = inside(data_root, relative)
        if not source.is_file():
            raise FileNotFoundError(f"Required bundle missing from Drive: {source}")
        expected = (expected_archives or {}).get(relative)
        if expected and source.stat().st_size != expected["bytes"]:
            raise ValueError(f"Size differs from the Option A manifest: {source.name}")
        staged = scratch / f"{name}-{source.name}"
        shutil.copyfile(source, staged)
        checksum = digest(staged)
        if expected and checksum != expected["sha256"]:
            raise ValueError(f"SHA-256 differs from the Option A manifest: {source.name}")
        root = scratch / name
        root.mkdir()
        extract_bundle(staged, root)
        staged.unlink()
        capture_meta = root / "capture.json"
        if capture_meta.is_file():
            meta = json.loads(capture_meta.read_text(encoding="utf-8"))
            if meta.get("purpose") != "TRAIN_ONLY" or val_manifest:
                raise ValueError(f"Capture bundle is not designated training-only: {source.name}")
            train_groups.update(meta.get("groups", []))
        train_rows = load_sft(root, train_manifest, name, image_hashes, training=True)
        val_rows = load_sft(root, val_manifest, name, image_hashes, training=False) if val_manifest else []
        if set(row[1] for row in train_rows) & set(row[1] for row in val_rows):
            raise ValueError(f"Train/validation image overlap in {name}")
        train_rows, duplicates = deduplicate(sorted(train_rows, key=lambda row: row[2]["sample_id"]))
        if train_cap is not None:
            if type(train_cap) is not int or train_cap < 1:
                raise ValueError("train_cap must be a positive integer or None")
            train_rows = rng.sample(train_rows, min(train_cap, len(train_rows)))
        training.extend(train_rows)
        validation.extend(val_rows)
        receipt[name] = {"archive": relative, "sha256": checksum, "train_manifest": train_manifest,
                         "validation_manifest": val_manifest, "train_actions": dict(Counter(row[0] for row in train_rows)),
                         "validation_actions_before_cap": dict(Counter(row[0] for row in val_rows)),
                         "training_deduplication": duplicates}
    if not training or not validation:
        raise ValueError("An explicit training split and an explicit development/validation split are required")
    if set(row[1] for row in training) & set(row[1] for row in validation):
        raise ValueError("Train/validation image hashes overlap across sources")
    training, train_merged_duplicates = deduplicate(training)
    validation, val_merged_duplicates = deduplicate(validation)
    if not training or not validation:
        raise ValueError("Train or development set is empty after removing conflicting duplicates")
    if dev_limit is not None:
        if type(dev_limit) is not int or dev_limit < 1:
            raise ValueError("dev_limit must be a positive integer or None")
        validation = rng.sample(validation, min(dev_limit, len(validation)))
    rng.shuffle(training)
    write_jsonl(scratch / "train.jsonl", (row[2] for row in training))
    write_jsonl(scratch / "validation.jsonl", (row[2] for row in validation))
    report = {"seed": seed, "sources": receipt, "train_count": len(training), "validation_count": len(validation),
              "train_actions": dict(Counter(row[0] for row in training)), "validation_actions": dict(Counter(row[0] for row in validation)),
              "merged_deduplication": {"training": train_merged_duplicates, "validation": val_merged_duplicates},
              "train_groups_in_capture_manifests": sorted(train_groups),
              "train_hashes": sorted({row[1] for row in training}), "validation_hashes": sorted({row[1] for row in validation})}
    write_json(scratch / "data_receipt.json", report)
    return report


def install_colab_stack(install_cuda_accelerators):
    pip = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
    subprocess.run(pip + ["--index-url", "https://download.pytorch.org/whl/cu130", "torch==2.14.1+cu130",
                          "torchvision==0.29.1+cu130"], check=True)
    subprocess.run(pip + ["ms-swift==4.6.1", "transformers==5.22.2", "qwen-vl-utils==0.0.14", "peft==0.20.0",
                          "torchao==0.18.0", "Pillow==11.3.0", "flash-linear-attention==0.6.0"], check=True)
    if install_cuda_accelerators:
        subprocess.run(pip + ["--no-build-isolation", "git+https://github.com/Dao-AILab/causal-conv1d@cd81f0413cad2fc1e6f17e785ac39f59aae690cd"], check=True)
    imports = "import swift, transformers, torch, torchvision, peft, torchao; from peft.import_utils import is_torchao_available; assert is_torchao_available()"
    subprocess.run([sys.executable, "-c", imports], check=True)


def install_cpu_export_stack():
    from importlib.metadata import PackageNotFoundError, version

    packages = {"ms-swift": "4.6.1", "transformers": "5.22.2", "peft": "0.20.0",
                "qwen-vl-utils": "0.0.14", "torchao": "0.18.0"}
    requirements = []
    for name, pinned in packages.items():
        try:
            if version(name) == pinned:
                continue
        except PackageNotFoundError:
            pass
        requirements.append(f"{name}=={pinned}")
    if requirements:
        subprocess.run([sys.executable, "-m", "pip", "install", *requirements], check=True)


def checkpoints(output):
    return sorted((path for path in output.rglob("checkpoint-*") if (path / "adapter_model.safetensors").is_file()),
                  key=lambda path: int(path.name.removeprefix("checkpoint-")))


def run_training(command, environment, scratch, run_dir):
    gpu_check = "import torch; assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(); print(torch.cuda.get_device_name(0))"
    subprocess.run([sys.executable, "-c", gpu_check], check=True)
    run_dir.mkdir(parents=True, exist_ok=False)
    shutil.copyfile(scratch / "data_receipt.json", run_dir / "data_receipt.json")
    write_json(run_dir / "training_config.json", {"command": command, "environment": environment,
                                                 "started_at": datetime.now(timezone.utc).isoformat()})
    with (run_dir / "pip_freeze.txt").open("w", encoding="utf-8") as stream:
        subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=stream, check=True)
    with (run_dir / "sft.log").open("w", encoding="utf-8", buffering=1) as log:
        with subprocess.Popen(command, env={**os.environ, **environment}, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            if process.wait():
                raise RuntimeError(f"SFT failed; inspect {run_dir / 'sft.log'}")
    available = checkpoints(run_dir / "checkpoints")
    if not available:
        raise RuntimeError("No complete LoRA checkpoint was written")
    return available[-1]


def require_a100_guard():
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Switch Runtime -> Change runtime type -> A100 GPU, then Run all")
    probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    hardware = probe.stdout.strip()
    if probe.returncode != 0 or "A100" not in hardware:
        raise RuntimeError(f"Switch Runtime -> Change runtime type -> A100 GPU, then Run all (found: {hardware!r})")
    print(f"[OK] GPU confirmed: {hardware}")


def swift_command_m35(scratch, output, *, model_revision, seed, rank=32, batch=4, accumulation=8,
                      image_tokens=1024, learning_rate=1e-4, epochs=2, cadence=50):
    if len(model_revision) != 40 or any(char not in "0123456789abcdef" for char in model_revision):
        raise ValueError("model_revision must be an exact 40-char Hugging Face commit")
    command = [
        "swift", "sft", "--model", BASE_MODEL, "--model_revision", model_revision, "--use_hf", "true",
        "--tuner_type", "lora", "--target_modules", "all-linear",
        "--lora_rank", str(rank), "--lora_alpha", str(rank * 2), "--lora_dropout", "0.05",
        "--freeze_vit", "true", "--freeze_aligner", "true", "--freeze_llm", "false",
        "--add_non_thinking_prefix", "true", "--loss_scale", "ignore_empty_think",
        "--dataset", str(scratch / "train.jsonl"), "--val_dataset", str(scratch / "validation.jsonl"),
        "--split_dataset_ratio", "0", "--torch_dtype", "bfloat16", "--attn_impl", "sdpa",
        "--per_device_train_batch_size", str(batch), "--per_device_eval_batch_size", str(batch),
        "--gradient_accumulation_steps", str(accumulation), "--learning_rate", str(learning_rate),
        "--lr_scheduler_type", "cosine", "--warmup_ratio", "0.05",
        "--num_train_epochs", str(epochs), "--max_length", "4096",
        "--eval_steps", str(cadence), "--save_steps", str(cadence), "--save_total_limit", "6",
        "--logging_steps", "1", "--dataloader_num_workers", "0", "--seed", str(seed),
        "--output_dir", str(output),
    ]
    env = {
        "IMAGE_MAX_TOKEN_NUM": str(image_tokens),
        "PYTHONUNBUFFERED": "1",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "CUDA_VISIBLE_DEVICES": "0",
    }
    return command, env


def export_gguf_m35(adapter, run_dir, scratch, revision, device="cpu"):
    if len(revision) != 40 or any(char not in "0123456789abcdef" for char in revision):
        raise ValueError("Pin the verified llama.cpp commit before export")
    if not (adapter / "adapter_config.json").is_file() or not (adapter / "adapter_model.safetensors").is_file():
        raise ValueError(f"Export requires a complete PEFT LoRA checkpoint: {adapter}")
    if device not in {"cpu", "auto"}:
        raise ValueError("Export device must be cpu or auto")
    config = json.loads((adapter / "adapter_config.json").read_text(encoding="utf-8"))
    if config.get("base_model_name_or_path", "").split("/")[-1] not in {BASE_MODEL.rsplit("/", 1)[-1], BASE_REVISION}:
        raise ValueError("Adapter does not identify the expected Qwen base")

    root = scratch / "llama.cpp"
    build = root / "build"
    merged = scratch / "merged"
    bundle = scratch / "gguf"
    for p in (root, merged, bundle):
        if p.exists():
            shutil.rmtree(p)
    bundle.mkdir(parents=True)

    subprocess.run([
        "swift", "export", "--adapters", str(adapter), "--use_hf", "true", "--merge_lora", "true",
        "--model", BASE_MODEL, "--model_revision", BASE_REVISION, "--device_map", device,
        "--torch_dtype", "bfloat16", "--output_dir", str(merged),
    ], check=True)

    root.mkdir()
    subprocess.run(["git", "init", str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "fetch", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", revision], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gguf", "sentencepiece"], check=True)
    subprocess.run([
        "cmake", "-S", str(root), "-B", str(build), "-DGGML_CUDA=OFF", "-DLLAMA_BUILD_TESTS=OFF",
        "-DLLAMA_BUILD_SERVER=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF", "-DLLAMA_BUILD_TOOLS=ON",
    ], check=True)
    subprocess.run(["cmake", "--build", str(build), "--target", "llama-quantize", "-j4"], check=True)

    converter = [sys.executable, str(root / "convert_hf_to_gguf.py"), str(merged)]
    bf16 = bundle / "m35-bf16.gguf"
    projector = bundle / "m35-mmproj-f16.gguf"
    subprocess.run(converter + ["--outfile", str(bf16), "--outtype", "bf16"], check=True)
    subprocess.run(converter + ["--mmproj", "--outfile", str(projector), "--outtype", "f16"], check=True)

    files = [projector]
    for quant in ("Q8_0", "Q4_K_M"):
        target = bundle / f"m35-qwen35-2b-{quant}.gguf"
        subprocess.run([str(build / "bin" / "llama-quantize"), str(bf16), str(target), quant], check=True)
        files.append(target)
    if bf16.is_file():
        bf16.unlink()

    sha_map = {}
    for path in files:
        with path.open("rb") as stream:
            if stream.read(4) != b"GGUF":
                raise ValueError(f"Invalid GGUF: {path.name}")
        sha_map[path.name] = digest(path)

    output = run_dir / "gguf"
    output.mkdir(parents=True, exist_ok=True)
    for path in files:
        dst = output / path.name
        shutil.copyfile(path, dst)
        if digest(dst) != sha_map[path.name]:
            raise RuntimeError(f"Drive copy SHA-256 verification failed: {dst}")

    write_json(output / "export_receipt.json", {
        "revision": revision,
        "model_revision": BASE_REVISION,
        "adapter": str(adapter),
        "deployment_validated": False,
        "sha256": sha_map,
    })

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/chrome-gui-pilot")
RUN_ID = datetime.now(timezone.utc).strftime("m35-%Y%m%d-%H%M%S") + "-sft2ep"
SEED = 42
MODEL_REVISION = BASE_REVISION
LLAMA_CPP_REVISION = LLAMA_CPP_EXPORT_REVISION

RUN_TRAINING = True
RUN_EXPORT = True
EXPORT_ADAPTER = None
INSTALL_STACK = True
INSTALL_CUDA_ACCELERATORS = False
AUTO_UNASSIGN_ON_FINISH = True
EXPORT_ONLY = RUN_EXPORT and not RUN_TRAINING

SOURCES = [
    ("m1_web", "m1_web_bundle.tar", "train_open_web.jsonl", "val_open_web.jsonl", 1500),
    ("zh_fixture", "M3_zh_chrome_fixture_train_60.zip", "train_chrome.jsonl", None, None),
    ("satraj_chrome", "m3_satraj_chrome.tar", "train_satraj_chrome.jsonl", "val_satraj_chrome.jsonl", None),
    ("zh_boost", "m3_option_a_zh_boost.tar", "train_zh_boost.jsonl", None, None),
    ("hard_neg", "m3_option_a_hard_neg.tar", "train_hard_neg.jsonl", None, 300),
    ("m35_real_chrome", "m35_real_chrome_train_dev.tar", "train_real_chrome.jsonl", "val_real_chrome.jsonl", None),
]

EXPECTED_ARCHIVES = {
    "m3_satraj_chrome.tar": {
        "bytes": 152770560,
        "sha256": "6e364a5d0befc45d0fec89cddac121eb4dfae2da5dd495170eabe8599a652dad",
    },
    "m3_option_a_zh_boost.tar": {
        "bytes": 4577280,
        "sha256": "7b65ddfa185ca529fe4b49aadbbc5cf813b19102820dce280c6201f2e5242937",
    },
    "m3_option_a_hard_neg.tar": {
        "bytes": 77178880,
        "sha256": "2ed796d2bdb27c8d61140a8e54afc7d3582417c649ec00c0f41f0697832a85d9",
    },
    "m35_real_chrome_train_dev.tar": {
        "bytes": 10465280,
        "sha256": "0ecd6dea3352735753d59a4017636022f762a31cda43bc85caf8a18f34253b4d",
    },
}

In [ ]:
a100_confirmed = False
WORK = Path("/content/chrome-pilot-m35") / RUN_ID
RUN_DIR = DRIVE_ROOT / "m35_runs" / RUN_ID

try:
    if RUN_TRAINING:
        require_a100_guard()
        a100_confirmed = True
        if RUN_DIR.exists():
            raise FileExistsError(f"Choose a fresh RUN_ID; already exists: {RUN_DIR}")
    elif RUN_EXPORT and not EXPORT_ADAPTER:
        raise ValueError("Set EXPORT_ADAPTER to a saved PEFT checkpoint when RUN_TRAINING=False")

    if not EXPORT_ONLY:
        if WORK.exists():
            shutil.rmtree(WORK)
        report = prepare(
            DRIVE_ROOT / "data",
            WORK,
            SOURCES,
            SEED,
            dev_limit=None,
            expected_archives=EXPECTED_ARCHIVES,
        )
        print(json.dumps({k: v for k, v in report.items() if not k.endswith("hashes")}, ensure_ascii=False, indent=2), flush=True)
        command, run_env = swift_command_m35(WORK, RUN_DIR / "checkpoints", model_revision=MODEL_REVISION, seed=SEED)
        print("Command:", " ".join(command), flush=True)

    if INSTALL_STACK:
        if EXPORT_ONLY:
            install_cpu_export_stack()
        else:
            install_colab_stack(INSTALL_CUDA_ACCELERATORS)

    adapter = None
    if RUN_TRAINING:
        RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
        adapter = run_training(command, run_env, WORK, RUN_DIR)
        print(f"[OK] Final LoRA checkpoint: {adapter}", flush=True)

    if RUN_EXPORT:
        WORK.mkdir(parents=True, exist_ok=True)
        if EXPORT_ONLY:
            RUN_DIR.mkdir(parents=True, exist_ok=True)
        target_adapter = adapter if adapter is not None else Path(EXPORT_ADAPTER)
        export_gguf_m35(target_adapter, RUN_DIR, WORK, LLAMA_CPP_REVISION, device="cpu")
        print(f"[OK] GGUF artifacts exported & verified at: {RUN_DIR / 'gguf'}", flush=True)
except Exception:
    err_text = traceback.format_exc()
    print(err_text, flush=True)
    if RUN_DIR.is_dir():
        (RUN_DIR / "error.txt").write_text(err_text, encoding="utf-8")
    raise
finally:
    try:
        os.sync()
    except Exception:
        pass
    time.sleep(12)
    if AUTO_UNASSIGN_ON_FINISH and a100_confirmed:
        print("[CLEANUP] Syncing Drive and unassigning A100 runtime...", flush=True)
        from google.colab import runtime
        runtime.unassign()